In [1]:
# Libraries

import os
import pandas as pd

In [ ]:
# Download the data
os.chdir("/Users/jananidhileepan/Desktop/Don't. Even./University/Imperial College London/Year 2/Dissertation/GitHub/fairness_smoke_alarm/data/raw/hmda") # To go back to the root directory of the project
os.getcwd()

df_raw = pd.read_csv("hmda_2007_ca_all-records_labels.csv")

df_raw.shape # shape: (3425570, 78)

/var/folders/jk/86ps_fvj28106lt4y2bq856m0000gn/T/ipykernel_75736/1296412443.py:5: DtypeWarning: Columns (34,36,38,44,46,48,57,59,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("hmda_2007_ca_all-records_labels.csv")


(3425570, 78)

In [3]:
# Keep relevant columns
cols_to_keep = ["as_of_year", "loan_type_name", "loan_type", "loan_amount_000s", 
                "action_taken_name", "action_taken", "county_name", "county_code",
                  "applicant_ethnicity_name", "applicant_ethnicity", "co_applicant_ethnicity_name",
                  "co_applicant_ethnicity", "applicant_race_name_1", "applicant_race_1",
                  "applicant_race_name_2", "applicant_race_2", "applicant_race_name_3",
                  "applicant_race_3", "applicant_race_name_4", "applicant_race_4",
                    "applicant_race_name_5", "applicant_race_5", "co_applicant_race_name_1",
                    "co_applicant_race_1", "co_applicant_race_name_2", "co_applicant_race_2",
                    "co_applicant_race_name_3", "co_applicant_race_3", "co_applicant_race_name_4", "co_applicant_race_4",
                    "co_applicant_race_name_5", "co_applicant_race_5", "applicant_sex_name",
                    "applicant_sex", "co_applicant_sex_name", "co_applicant_sex",
                    "applicant_income_000s", "denial_reason_name_1", "denial_reason_1",
                    "denial_reason_name_2", "denial_reason_2", "denial_reason_name_3",
                    "denial_reason_3", "lien_status_name", "lien_status",
                    "tract_to_msamd_income"]

df = df_raw[cols_to_keep]

In [4]:
# Check missingness by column
missing_df = df.isnull().sum(axis=0)
print(missing_df)

as_of_year                           0
loan_type_name                       0
loan_type                            0
loan_amount_000s                     0
action_taken_name                    0
action_taken                         0
county_name                       7009
county_code                       7009
applicant_ethnicity_name             0
applicant_ethnicity                  0
co_applicant_ethnicity_name          0
co_applicant_ethnicity               0
applicant_race_name_1                0
applicant_race_1                     0
applicant_race_name_2          3410247
applicant_race_2               3410247
applicant_race_name_3          3424790
applicant_race_3               3424790
applicant_race_name_4          3425272
applicant_race_4               3425272
applicant_race_name_5          3425340
applicant_race_5               3425340
co_applicant_race_name_1             0
co_applicant_race_1                  0
co_applicant_race_name_2       3419783
co_applicant_race_2      

Nothing worrying aside from applicant_income_000s. county_code and tract_to_msamd_income are negligibly low. Fill in applicant_income_000s with caution

In [5]:
df.groupby("applicant_ethnicity_name").size()

applicant_ethnicity_name
Hispanic or Latino                                                                    850653
Information not provided by applicant in mail, Internet, or telephone application     553654
Not Hispanic or Latino                                                               1675236
Not applicable                                                                        346027
dtype: int64

In [6]:
df.groupby("applicant_race_name_1").size()

applicant_race_name_1
American Indian or Alaska Native                                                       71316
Asian                                                                                 291648
Black or African American                                                             162937
Information not provided by applicant in mail, Internet, or telephone application     625243
Native Hawaiian or Other Pacific Islander                                              43648
Not applicable                                                                        342358
White                                                                                1888420
dtype: int64

In [7]:
df.groupby("applicant_sex_name").size()

applicant_sex_name
Female                                                                                952285
Information not provided by applicant in mail, Internet, or telephone application     255268
Male                                                                                 1874278
Not applicable                                                                        343739
dtype: int64

In [8]:
df.groupby("action_taken_name").size()

action_taken_name
Application approved but not accepted                   324625
Application denied by financial institution             801501
Application withdrawn by applicant                      316025
File closed for incompleteness                          106510
Loan originated                                        1233502
Loan purchased by the institution                       642852
Preapproval request approved but not accepted               36
Preapproval request denied by financial institution        519
dtype: int64

In [9]:
def true_missingness(col):
    return (df[col].isnull().sum() + (df[col] == "Not applicable").sum() + (df[col] == "Information not provided by applicant in mail, Internet, or telephone application").sum())/len(df)

print("Missing applicant ethnicity name:", true_missingness("applicant_ethnicity_name"))
print("Missing applicant race name:", true_missingness("applicant_race_name_1"))
print("Missing applicant sex name:", true_missingness("applicant_sex_name"))

Missing applicant ethnicity name: 0.26263687503101674
Missing applicant race name: 0.28246423222996464
Missing applicant sex name: 0.17486345338148104


Check if "Loan purchased by the institution" is the driving factor behind missingness

In [10]:
df_test = df[df.action_taken != 6] # Remove pre-approvals, which are not relevant for our analysis

In [11]:
def true_missingness_test(col):
    return (df_test[col].isnull().sum() + (df_test[col] == "Not applicable").sum() + (df_test[col] == "Information not provided by applicant in mail, Internet, or telephone application").sum())/len(df_test)

print("Missing applicant ethnicity name:", true_missingness_test("applicant_ethnicity_name"))
print("Missing applicant race name:", true_missingness_test("applicant_race_name_1"))
print("Missing applicant sex name:", true_missingness_test("applicant_sex_name"))

Missing applicant ethnicity name: 0.1869973888838179
Missing applicant race name: 0.21030445772801987
Missing applicant sex name: 0.08985315795563906
